![MuJoCo Playground banner](https://www.gstatic.com/reaf-sim/banner/0/banner.png)

# MuJoCo Playground & `reaf-sim` Quickstart <a href="https://colab.research.google.com/github/google-deepmind/mujoco_playground/blob/main/notebooks/quickstart.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" align="center"/></a>

[MuJoCo Playground](https://github.com/google-deepmind/mujoco_playground) and [reaf-sim](https://github.com/google-deepmind/reaf-sim) are PyTorch simulation frameworks built by the MuJoCo team.

This notebook runs on **GPU** (recommended: *Runtime > Change runtime type > GPU*) or **CPU**, and walks through:
1. **Constructing a `reaf-sim` environment** from scratch (batched Cartpole with camera rendering).
2. **Training `CartpoleBalance` with PPO** (RSL-RL) and rendering the trained policy.
3. **Training `SO101PickCube` with PPO** (RSL-RL) and rendering the trained policy.

## Setup

In [ ]:
#@title Setup
import dataclasses, json, os, subprocess, sys

# Uninstall Colab's preinstalled TensorFlow so its LLVM does not collide with Warp/Mesa.
if os.environ.get("COLAB_RELEASE_TAG"):
  subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "tensorflow"], check=False)

!uv pip install --system -q "playground[rl]>=1.1" mediapy

import torch
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda:0" if HAS_GPU else "cpu"

if HAS_GPU and os.geteuid() == 0:
  icd = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
  os.makedirs(os.path.dirname(icd), exist_ok=True)
  if not os.path.exists(icd):
    open(icd, "w").write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')
elif not HAS_GPU:
  os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"
os.environ["MUJOCO_GL"] = "egl"

import mediapy as media, mujoco, numpy as np
print(f"Using device={DEVICE}, MUJOCO_GL=egl, torch={torch.__version__}, mujoco={mujoco.__version__}")

## 1. Construct a `reaf-sim` Environment (Cartpole)

Environments in `reaf-sim` are built from a **declarative config** (`flat_scene.Environment`): you describe the MuJoCo `base_scene` (`MjSpec`), simulator settings (`Sim`), hardware/sensor components (`DeviceLayer`), and observation/reward/termination logic (`observation_adapter` and `TaskLayer`) as plain Python dataclasses, then call `flat_scene.build_env(config)` to compile the model and instantiate the environment.

Because `DeviceLayer` (hardware/sensor I/O) is decoupled from `TaskLayer` (MDP logic with no access to privileged sim state), the same task config runs in simulation and on real robots. Below, we declare a batched Cartpole environment with `ActuatorDevice`, `ArticulatedDevice`, and `CameraDevice` on MuJoCo Warp (`impl="warp"`), step with random actions, and render a tiled video.

In [ ]:
from reaf.simulation.torch import ActuatorDevice, ArticulatedDevice, CameraDevice, flat_scene
from reaf.simulation.torch.examples.cartpole import CARTPOLE_XML, tile_grid

nworld = 16 if HAS_GPU else 4
spec = mujoco.MjSpec.from_string(CARTPOLE_XML)

config = flat_scene.Environment(
    base_scene=spec,
    sim=flat_scene.Sim(
        nworld=nworld,
        xpu_device=DEVICE,
        impl="warp",
        simulation_timestep=0.01,
        control_timestep=0.05,
    ),
    device_layer=flat_scene.DeviceLayer(
        devices={
            "slider": flat_scene.Component(ActuatorDevice, targets=["slider"]),
            "joints": flat_scene.Component(
                ArticulatedDevice,
                joint_names=["slider", "hinge_1"],
                measurements=["joint_pos", "joint_vel"],
            ),
            "camera": flat_scene.Component(
                CameraDevice,
                cam_name="fixed",
                cam_resolution=(160, 120),
                render_rate_hz=20.0,
                output_name_prefix="camera/",
            ),
        }
    ),
    observation_adapter={
        "state": flat_scene.ObsGroup(
            terms={
                "qpos": flat_scene.Obs("joints/joint_pos").configure(output_shape=(2,)),
                "qvel": flat_scene.Obs("joints/joint_vel").configure(output_shape=(2,)),
            }
        ),
        "camera/fixed_rgb": flat_scene.ObsGroup(
            terms={
                "rgb": flat_scene.Obs("camera/fixed_rgb").configure(
                    output_shape=(120, 160, 3), dtype="uint8"
                ),
            }
        ),
    },
)

env = flat_scene.build_env(config)
timestep = env.reset()
print("State observation shape:", tuple(timestep.observation["state"].shape))

cols = 4 if nworld >= 4 else nworld
frames = []
for _ in range(100):
  commands = (torch.rand((env.nworld, 1), device=DEVICE) * 2.0 - 1.0) * 5.0
  timestep = env.step({"slider/ctrl": commands})
  frames.append(tile_grid(timestep.observation["camera/fixed_rgb"], cols=cols))

media.show_video(frames, fps=20)

### Helper: Train & Render a Trained PPO Policy

In [ ]:
from mujoco_playground.rl import env_registry
from mujoco_playground._src.rl.rsl_rl import checkpoint as rsl_checkpoint
from mujoco_playground._src.rl.rsl_rl import vecenv_wrapper
from rsl_rl import runners as rsl_rl_runners

def train_ppo(env_name: str, logdir: str, nworld: int, max_iterations: int) -> None:
  overrides = json.dumps({"nworld": nworld, "max_iterations": max_iterations})
  cmd = [
      sys.executable,
      "-m",
      "mujoco_playground.rl.rsl_rl.train",
      f"--env={env_name}",
      f"--logdir={logdir}",
      f"--device={DEVICE}",
      f"--overrides={overrides}",
  ]
  subprocess.run(cmd, check=True)

def render_policy(
    env_name: str,
    logdir: str,
    num_steps: int,
    camera: str | int,
) -> tuple[list[np.ndarray], float]:
  # Build a single-world env and load the latest trained RSL-RL policy.
  entry = env_registry.get(env_name)
  agent_cfg = dataclasses.replace(entry.runner_cfg, nworld=1)
  raw_env = entry.env_builder(nworld=1, device=DEVICE)
  vec_env = vecenv_wrapper.RslRlVecEnvWrapper(
      raw_env,
      clip_actions=agent_cfg.clip_actions,
      cfg=agent_cfg.vecenv_wrapper_cfg,
  )
  runner = rsl_rl_runners.OnPolicyRunner(
      vec_env, dataclasses.asdict(agent_cfg), device=DEVICE
  )
  runner.load(rsl_checkpoint.read_checkpoint(logdir), map_location=DEVICE)
  policy = runner.get_inference_policy(device=DEVICE)

  # Set up an offscreen MuJoCo renderer backed by a CPU MjData copy.
  sim = raw_env.device_layer.sim
  mj_model = sim.mj_model
  mj_data = mujoco.MjData(mj_model)
  renderer = mujoco.Renderer(mj_model, height=240, width=320)

  # Roll out the policy and render each control step.
  obs, _ = vec_env.reset()
  frames = []
  for _ in range(num_steps):
    with torch.no_grad():
      action = policy(obs)
    obs, _, _, _ = vec_env.step(action)
    mj_data.qpos[:] = sim.data.qpos[0].detach().cpu().numpy()
    mj_data.qvel[:] = sim.data.qvel[0].detach().cpu().numpy()
    mujoco.mj_forward(mj_model, mj_data)
    renderer.update_scene(mj_data, camera=camera)
    frames.append(renderer.render().copy())
  renderer.close()
  fps = max(1, round(1.0 / raw_env.device_layer.control_timestep))
  return frames, fps

## 2. Train Cartpole with PPO (`CartpoleBalance`)

Train the registered `CartpoleBalance` task with RSL-RL PPO and visualize the learned policy.

In [ ]:
cartpole_logdir = "/tmp/rslrl_logs/cartpole_balance"
train_ppo(
    "CartpoleBalance",
    logdir=cartpole_logdir,
    nworld=2048 if HAS_GPU else 64,
    max_iterations=100 if HAS_GPU else 10,
)

frames, fps = render_policy("CartpoleBalance", cartpole_logdir, num_steps=200, camera=-1)
media.show_video(frames, fps=fps)

## 3. Train Pick Cube with PPO (`SO101PickCube`)

MuJoCo Playground includes a vision-based `SO101PickCubeVision` task that transfers zero-shot from simulation to the physical SO-101 arm (`--mode=real`) using the same `DeviceLayer`/`TaskLayer` split. Here we train the faster proprio-only `SO101PickCube` variant (which uses privileged cube state) for illustrative purposes in Colab.

In [ ]:
pick_cube_logdir = "/tmp/rslrl_logs/so101_pick_cube"
train_ppo(
    "SO101PickCube",
    logdir=pick_cube_logdir,
    nworld=2048 if HAS_GPU else 64,
    max_iterations=300 if HAS_GPU else 5,
)

frames, fps = render_policy("SO101PickCube", pick_cube_logdir, num_steps=200, camera=-1)
media.show_video(frames, fps=fps)

## Next Steps

- **Documentation:** [reaf-sim & MuJoCo Playground docs](https://reaf-sim.readthedocs.io/en/latest/)
- **RL Training:** [RSL-RL training guide](https://reaf-sim.readthedocs.io/en/latest/playground/rl_training.html)
- **Sim-to-Real:** [Teleoperation, sysid, and hardware deployment](https://reaf-sim.readthedocs.io/en/latest/playground/sim2real.html)